In [1]:
from PyPDF2 import PdfReader
import os
import re
import json

pdfs_dir = r"C:/Users/TP_USER/Downloads/PDFTEXTS/"
train_data = []
for folder in os.listdir(pdfs_dir):
    pdf_dir = os.path.join(pdfs_dir, folder)
    for pdf_name in os.listdir(pdf_dir):
            if pdf_name.lower().endswith("json") and folder == 'Aditya':
            # Write text to the file
                with open(os.path.join(pdf_dir, pdf_name), "r", encoding="utf-8") as f:
                    data = json.load(f)
                train_data.append(data['annotations'][0])


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:/Users/TP_USER/Downloads/PDFTEXTS/'

In [75]:
import spacy
import json
from spacy.training import offsets_to_biluo_tags
import os

def check_data(nlp, train_data):
    flag = False
    for text, entities in train_data:
        # print(text, entities)
        tags = offsets_to_biluo_tags(nlp.make_doc(text), entities['entities'])
        if "-" in tags:
            print(f"Tags for '{text}' --> {tags}")
            flag = True
    if flag:
        print("Mislabelled Training Data")
    else:
        print("Correctly labelled training data")
        
# pdfs_dir = r"C:/Users/TP_USER/Downloads/PDFTEXTS/"
# train_data = []
# for folder in os.listdir(pdfs_dir):
#     pdf_dir = os.path.join(pdfs_dir, folder)
#     for pdf_name in os.listdir(pdf_dir):
#             if pdf_name.lower().endswith("json"):
#             # Write text to the file
#                 with open(os.path.join(pdf_dir, pdf_name), "r", encoding="utf-8") as f:
#                     data = json.load(f)
#                 train_data.append(data['annotations'][0])
nlp = spacy.load("en_core_web_sm")

# file_path = r"C:\Users\TP_USER\Downloads\Avnue.json"
# with open(file_path, "r", encoding='utf-8') as f:
#     entities = json.load(f)
#     check_data(nlp, entities)
train_data = []
directory = r"C:\Users\TP_USER\Downloads\PDFTEXTS\Max"
for annot_file in os.listdir(directory):
    if annot_file.endswith('json') and annot_file.startswith("annot"):
        annot_file_path = os.path.join(directory, annot_file)
        with open(annot_file_path, "r", encoding='utf-8') as f:
            data = json.load(f)
            for text, entities in data['annotations']:
#                 print(entities)
                train_data.append([text, entities])
#             check_data(nlp, data['annotations'])

In [77]:
with open("max.json", "w", encoding='utf-8') as f:
    json.dump(train_data, f, indent=4)

In [71]:
import os
import json

def find_nth_occurrence(text, word, n):
    occurrence_count = 0
    occurrence_index = -1
    
    while occurrence_count < n:
        occurrence_index = text.find(word, occurrence_index + 1)
        if occurrence_index == -1:
            return -1
        occurrence_count += 1
        
    return occurrence_index

annotated_in = r"C:\Users\TP_USER\Downloads\max_annotations\annotated"
annotated_new = r"C:\Users\TP_USER\Downloads\max_annotations\annotated_new"
data_freq_list = []
for annot_file in os.listdir(annotated_in):
    if annot_file.endswith("json"):
        annot_file_path = os.path.join(annotated_in, annot_file)
        with open(annot_file_path, "r", encoding='utf-8') as f:
            data_read = json.load(f)
            text, annots = data_read['annotations'][0]
            data_freq = {}
            for annot in annots['entities']:
                start, end, label = annot
                txt = text[start:end]
                if txt not in data_freq.keys():
                    data_freq[txt] = [0, label]
                data_freq[txt][0] += 1
            data_freq_list.append(data_freq)

texts = []
for txt_file in os.listdir(annotated_new):
    if txt_file.endswith("txt"):
        txt_file_path = os.path.join(annotated_new, txt_file)
        with open(txt_file_path, "r", encoding='utf-8') as f:
            text = f.read()
            texts.append(text)
        

for index, (freq_dict, text)  in enumerate(zip(data_freq_list, texts)):
    entities = []
    for key, (value, label) in freq_dict.items():
        for occurance in range(value):
            start = find_nth_occurrence(text, key, occurance+1)
            end = start + len(key) - 1
            entities.append([start, end, label])
    classes = ['ORG', 'ADDRESS', 'GSTIN', 'PO NUMBER', 'PO DATE', "DELIVERY DATE", 'PO EXPIRY DATE', 'SUPPLIER NUMBER', 'PRODUCT DESCRIPTION', "PAN", "CIN", "QTY", "MRP"]
    annotation = {"classes":[cls for cls in classes],"annotations":[]}
    annotation['annotations'].append([text,{"entities": entities}])
    with open(f"{annotated_new}/PDF_{str(index+1).zfill(3)}.json", "w", encoding='utf-8') as annot_file:
        json.dump(annotation, annot_file)
    
    print(annotation['annotations'][0][1])

{'entities': [[0, 28, 'ORG'], [2780, 2808, 'ORG'], [100, 175, 'ADDRESS'], [2880, 2955, 'ADDRESS'], [205, 219, 'GSTIN'], [2985, 2999, 'GSTIN'], [-1, 104, 'ADDRESS'], [-1, 6, 'PO NUMBER'], [544, 548, 'SUPPLIER NUMBER'], [552, 577, 'ORG'], [597, 608, 'PO DATE'], [610, 664, 'ADDRESS'], [672, 686, 'GSTIN'], [704, 714, 'DELIVERY DATE'], [774, 784, 'PO EXPIRY DATE'], [792, 806, 'GSTIN'], [-1, 33, 'PRODUCT DESCRIPTION'], [-1, 33, 'PRODUCT DESCRIPTION']]}
{'entities': [[0, 28, 'ORG'], [5025, 5053, 'ORG'], [100, 175, 'ADDRESS'], [5125, 5200, 'ADDRESS'], [205, 219, 'GSTIN'], [5230, 5244, 'GSTIN'], [-1, 110, 'ADDRESS'], [-1, 6, 'PO NUMBER'], [-1, 3, 'SUPPLIER NUMBER'], [537, 562, 'ORG'], [582, 593, 'PO DATE'], [-1, 68, 'ADDRESS'], [-1, 13, 'GSTIN'], [689, 699, 'DELIVERY DATE'], [755, 765, 'PO EXPIRY DATE'], [-1, 13, 'GSTIN'], [-1, 22, 'PRODUCT DESCRIPTION'], [-1, 23, 'PRODUCT DESCRIPTION'], [-1, 23, 'PRODUCT DESCRIPTION'], [-1, 31, 'PRODUCT DESCRIPTION'], [-1, 37, 'PRODUCT DESCRIPTION']]}
{'entiti

In [82]:
import os
import json

annotated_dir_path = r"C:\Users\TP_USER\Downloads\format files\annotations_with_qty_and_mrp"
combined_data = []
for file in os.listdir(annotated_dir_path):
    file_path = os.path.join(annotated_dir_path, file)
    print(file_path)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        for single_data in data:
            combined_data.append(single_data)
check_data(nlp, combined_data)
with open("training_data_with_mrp_and_qty_except_reliance.json", "w", encoding='utf-8') as f:
    json.dump(combined_data, f, indent=4)

C:\Users\TP_USER\Downloads\format files\annotations_with_qty_and_mrp\aditya.json
C:\Users\TP_USER\Downloads\format files\annotations_with_qty_and_mrp\apollo.json
C:\Users\TP_USER\Downloads\format files\annotations_with_qty_and_mrp\avenue.json
C:\Users\TP_USER\Downloads\format files\annotations_with_qty_and_mrp\firstcry.json
C:\Users\TP_USER\Downloads\format files\annotations_with_qty_and_mrp\future.json
C:\Users\TP_USER\Downloads\format files\annotations_with_qty_and_mrp\max.json
C:\Users\TP_USER\Downloads\format files\annotations_with_qty_and_mrp\metro.json
C:\Users\TP_USER\Downloads\format files\annotations_with_qty_and_mrp\more.json
C:\Users\TP_USER\Downloads\format files\annotations_with_qty_and_mrp\optival.json
C:\Users\TP_USER\Downloads\format files\annotations_with_qty_and_mrp\vishal.json
C:\Users\TP_USER\Downloads\format files\annotations_with_qty_and_mrp\walmart.json
Correctly labelled training data


In [41]:
# program to categorize the pdfs

from PyPDF2 import PdfReader
import os
import shutil
import re
import fitz
import time

pdfs_dir = r"D:\Downloads\POReaderFormats\POReaderFormats"
save_dir = r"D:\Downloads\PO_format_pdfs"

for index, pdf_name in enumerate(os.listdir(pdfs_dir)):
    if pdf_name.lower().endswith("pdf"):
        pdf_path = os.path.join(pdfs_dir, pdf_name)
        pdf_read = fitz.open(pdf_path)
        text = ""
        for page in pdf_read:
            text += page.get_text("text")
        text = re.sub(r"\n", " ", text)
        text = re.sub(r"\s+", r" ", text)
        category = None
        if any(keyword.lower() in text.lower() for keyword in ["wal-mart", "walmart"]):
            category = "Walmart"
        elif "metro".lower() in text.lower():
            category = "Metro"
        elif "avenue".lower() in text.lower():
            category = "AvenueSupermarts"
        elif "Aditya".lower() in text.lower():
            category = "Aditya"
        elif "Metro".lower() in text.lower():
            category = "Metro"
        elif "Reliance".lower() in text.lower():
            category = "Reliance"
        elif "OPTIVAL HEALTH SOLUTIONS PVT LTD".lower() in text.lower():
            category = "Optival"
        elif "Apollo".lower() in text.lower():
            category = "Apollo"
        elif "HANDS ON TRADES PRIVATE LIMITED".lower() in text.lower():
            category = "Hands"
        elif "FSN DISTRIBUTION LIMITED".lower() in text.lower():
            category = "FSN"
        elif "Airplaza".lower() in text.lower():
            category = "Vishal"
        elif "more retail".lower() in text.lower():
            category = "More"
        elif "future retail".lower() in text.lower():
            category = "FutureRetail"
        elif "max hypermarket".lower() in text.lower():
            category = "Max"
        elif "pono: pin".lower() in text.lower():
            category = "FirstCry"
        elif "sap code".lower() in text.lower():
            category = "OnlyTable"
        
        if category:
            print(pdf_name)
            category_dir = os.path.join(save_dir, category)
            os.makedirs(category_dir, exist_ok=True)
            dest_path = os.path.join(category_dir, pdf_name)
            print(dest_path)
#             pdf_read.close()
#             time.sleep(1)
#             shutil.move(pdf_path, dest_path)
            print(f"Moved {pdf_name} to {category}")
        else:
            print(f"No category found for {pdf_name}")
            





No category found for 1715236586_98000073071420240509120654PM.pdf


In [17]:
# program to categorize the pdfs

from PyPDF2 import PdfReader
import os
import shutil
import re
import fitz
import time

pdfs_dir = r"D:\Downloads\PO_format_pdfs"
# pdfs_dir = r"D:\Downloads\POReaderFormats\POReaderFormats"
save_dir = r"D:\Downloads\PO_format_pdfs"

for folder in os.listdir(pdfs_dir):
    if folder.lower().endswith("pdf"):
        continue
    pdf_dir = os.path.join(pdfs_dir, folder)
    print()
    print(folder)
    for index, pdf_name in enumerate(os.listdir(pdf_dir)):
        if pdf_name.lower().endswith("pdf"):
            pdf_path = os.path.join(pdf_dir, pdf_name)
#             print("\n", pdf_path)
            pdf_read = fitz.open(pdf_path)
            text = ""
            for page in pdf_read:
                text += page.get_text("text")
            text = re.sub(r"\n", " ", text)
            text = re.sub(r"\s+", r" ", text)
            print(text[:120].strip())
            category = "Unknown"
            if any(keyword.lower() in text.lower() for keyword in ["wal-mart", "walmart"]):
                category = "Walmart"
            elif "Max Hypermarket India Pvt Ltd".lower() in text.lower():
                category = "Max"
            elif "Avenue Supermarts".lower() in text.lower() or "AvenueSupermarts".lower() in text.lower():
                category = "AvenueSupermarts"
            elif "more retail".lower() in text.lower():
                category = "More"
            elif "Aditya".lower() in text.lower():
                category = "Aditya"
            elif "Metro Cash & Carry India Private Limited".lower() in text.lower():
                category = "Metro"
            elif "Reliance".lower() in text.lower():
                category = "Reliance"
            elif "Optival".lower() in text.lower():
                category = "Optival"
            elif "Apollo".lower() in text.lower():
                category = "Apollo"
            elif "HANDS ON TRADES PRIVATE LIMITED".lower() in text.lower():
                category = "Hands"
            elif "FSN DISTRIBUTION LIMITED".lower() in text.lower():
                category = "FSN"
            elif "Airplaza".lower() in text.lower():
                category = "Vishal"
            elif "future retail".lower() in text.lower():
                category = "FutureRetail"
            elif "pono: ".lower() in text.lower():
                category = "FirstCry"
            elif "sap code".lower() in text.lower():
                category = "OnlyTable"
            elif "Metro Cash & Carry".lower() in text.lower():
                category = "Metro"
            elif "Policy Number".lower() in text.lower():
                category = "PNB"
            elif "Simple PDF File".lower() in text.lower():
                category = "SampleandBlank"

#             if category:
#                 category_dir = os.path.join(save_dir, category)
#                 os.makedirs(category_dir, exist_ok=True)
#                 dest_path = os.path.join(category_dir, pdf_name)
# #                 print(dest_path)
#                 pdf_read.close()
#                 time.sleep(0.1)
#                 shutil.move(pdf_path, dest_path)
#             else:
#                 print(f"No category found for {pdf_name}")
            





Aditya
Aditya Birla Retail Limited Regd. Office: Skyline Icon, 86/92, (5th and 6th Floor), Near Mittal Industrial Estate, Andhe
Aditya Birla Retail Limited Regd. Office: Skyline Icon, 86/92, (5th and 6th Floor), Near Mittal Industrial Estate, Andhe
Aditya Birla Retail Limited Regd. Office: Skyline Icon, 86/92, (5th and 6th Floor), Near Mittal Industrial Estate, Andhe
Aditya Birla Retail Limited Regd. Office: Skyline Icon, 86/92, (5th and 6th Floor), Near Mittal Industrial Estate, Andhe
Aditya Birla Retail Limited Regd. Office: Skyline Icon, 86/92, (5th and 6th Floor), Near Mittal Industrial Estate, Andhe
Aditya Birla Retail Limited Regd. Office: Skyline Icon, 86/92, (5th and 6th Floor), Near Mittal Industrial Estate, Andhe

Apollo
Delivery Address : Dag No-188,189,192,282,283, 285,286,289- 291,368-370,Kh No-2096 2097,JL No-83,Mouza-Dankuni Bill,Ward
Delivery Address : Dag No-188,189,192,282,283, 285,286,289- 291,368-370,Kh No-2096 2097,JL No-83,Mouza-Dankuni Bill,Ward
Delivery Address

Ship To Avenue Supermarts Ltd. Kompally Hyderabad DMart Hyderabad 500055 Phone Fax SY No 12&24, Ranga Reddy Email grn.ko
Ship To Avenue Supermarts Ltd. Ratlam Madhya Pradesh DMart Ratlam 457001 Phone Fax Old Jaawra By pass road Email grn.rat
Ship To Avenue Supermarts Ltd. Sihi Faridabad DMart Faridabad 121004 Phone Fax 75,Commercial Block-R,Sihi, Email grn.120
Ship To Avenue Supermarts Ltd. Seelapadi Dindigul Tamilnadu D Dindigul 624005 Phone Fax Trichy Bypass, Silapadi Village,
Ship To Avenue Supermarts Ltd. Seelapadi Dindigul Tamilnadu D Dindigul 624005 Phone Fax Trichy Bypass, Silapadi Village,
Ship To Avenue Supermarts Ltd. Seelapadi Dindigul Tamilnadu D Dindigul 624005 Phone Fax Trichy Bypass, Silapadi Village,
Ship To Avenue Supermarts Ltd. Seelapadi Dindigul Tamilnadu D Dindigul 624005 Phone Fax Trichy Bypass, Silapadi Village,
Ship To Avenue Supermarts Ltd. Seelapadi Dindigul Tamilnadu D Dindigul 624005 Phone Fax Trichy Bypass, Silapadi Village,
Ship To Avenue Supermarts Ltd. A

Ship To Avenue Supermarts Ltd. Attavar Road Mangalore DMart Dakshina Kannada 575001 Phone Fax 86 Attavar Road and NG Roa
Ship To Avenue Supermarts Ltd. Attavar Road Mangalore DMart Dakshina Kannada 575001 Phone Fax 86 Attavar Road and NG Roa
Ship To Avenue Supermarts Ltd. Attavar Road Mangalore DMart Dakshina Kannada 575001 Phone Fax 86 Attavar Road and NG Roa
Ship To Avenue Supermarts Ltd. Attavar Road Mangalore DMart Dakshina Kannada 575001 Phone Fax 86 Attavar Road and NG Roa
Ship To Avenue Supermarts Ltd. Attavar Road Mangalore DMart Dakshina Kannada 575001 Phone Fax 86 Attavar Road and NG Roa
Ship To Avenue Supermarts Ltd. Attavar Road Mangalore DMart Dakshina Kannada 575001 Phone Fax 86 Attavar Road and NG Roa
Ship To Avenue Supermarts Ltd. NH Derebail Mangalore Mangalore 575006 Phone 9892708818 Fax Nakkalgunta Road Email Buyer
Ship To Avenue Supermarts Ltd. Attavar Road Mangalore DMart Dakshina Kannada 575001 Phone Fax 86 Attavar Road and NG Roa
Ship To Avenue Supermarts Ltd. Ma

Ship To Avenue Supermarts Ltd. Uppillapalayam Coimbatore DMar Singanallur, Coimbatore 641015 Phone Fax Kamarajar Road, U
Ship To Avenue Supermarts Ltd. Uppillapalayam Coimbatore DMar Singanallur, Coimbatore 641015 Phone Fax Kamarajar Road, U
Ship To Avenue Supermarts Ltd. Coimbatore TN DMart Coimbatore South, Coimbatore 641023 Phone Fax 452 Podanur Main road,K
Ship To Avenue Supermarts Ltd. Omalur Road Salem DMart Salem 636004 Phone Fax Omalur Road Email grn.omalurroad@dmartindi
Ship To Avenue Supermarts Ltd. Sulur Coimbatore TN DMart Sulur 641402 Phone Fax Sulur Coimbatore Kannamalayam Email Buye
Ship To Avenue Supermarts Ltd. Uppillapalayam Coimbatore DMar Singanallur, Coimbatore 641015 Phone Fax Kamarajar Road, U
Ship To Avenue Supermarts Ltd. Coimbatore TN DMart Coimbatore South, Coimbatore 641023 Phone Fax 452 Podanur Main road,K
Ship To Avenue Supermarts Ltd. Tiruppur Tamil Nadu DMart Rakkiyapalayam,Tiruppur,Tamil Nadu. Phone Fax Aviinashi - Tirup
Ship To Avenue Supermarts Ltd. O

Min2Max Retail Company (Guindy - Chennai) Vendor Name: Himalaya Wellness Company(Chennai) PONO: pin101240506uyr1029 Addr
LACHMANDAS TRADING COMPANY (Kochi) Vendor Name: Himalaya Wellness Company (Ernakulam) PONO: pin142240508uwrdbfa Address:
LACHMANDAS TRADING COMPANY (Thiruvananthapuram) Vendor Name: Himalaya Wellness Company (Ernakulam) PONO: pin172240508xcc
RAKSHA FIRE AND SECURITY SYSTEMS Vendor Name: Himalaya Wellness Company(Bangalore) PONO: pin190240504qgyd215 Address: Ya
DS Enterprises(Patparganj - Delhi) Vendor Name: Himalaya Wellness Company(Delhi) PONO: pin192240506buv11d2 Address: 915-
Daiwick Corporation (Lucknow) Vendor Name: Himalaya Wellness Company(Lucknow) PONO: pin125240502tana8c7 Address: C-1/36,
Vikas Traders-Amritsar Vendor Name: Himalaya Wellness Company(Zikrapur) PONO: pin204240508aak7050 Address: Khata no 130/
Vignesh Enterprise -Kolkata Vendor Name: Himalaya Wellness Company(Kolkata) PONO: pin175240508cabe408 Address: NH-6, New
Sri Venkateshwara Enterprises Ve

...

Reliance Retail Limited C - 51,Plot No. 9, IDA Park, APIIC Road No. 8, Vakalpudi, Thammavaram Village Kakinada- 533005
Reliance Retail Limited C-135Phase -VIII Industrial Focal Point,Mohali, SAS Nagar- 160071 Punjab, INDIA SELLER PURCHASE
Reliance Retail Limited 1st Floor, Wing-A & B, Fortune Tower Chandrasekharpur,Bhubaneswar, Khurda- 751023 Odisha, INDIA
Reliance Retail Limited NO 62/2,RIL BUILIDING RICHMOND ROAD, BANGALORE- 560025 Karnataka, INDIA SELLER PURCHASE ORDER V
Reliance Retail Limited NO 62/2,RIL BUILIDING RICHMOND ROAD, BANGALORE- 560025 Karnataka, INDIA SELLER PURCHASE ORDER V
Reliance Retail Limited FF101,Saffron Panchwati 5 Rasta,, Near Bank of Baroda Ambawadi Ahmedabad- 380006 Gujarat, INDIA
Reliance Retail Limited FF101,Saffron Panchwati 5 Rasta,, Near Bank of Baroda Ambawadi Ahmedabad- 380006 Gujarat, INDIA
Reliance Retail Limited FF101,Saffron Panchwati 5 Rasta,, Near Bank of Baroda Ambawadi Ahmedabad- 380006 Gujarat, INDIA
Reliance Retail Limited 89, A1 Towers,Flo

Reliance Retail Limited 6- 3- 1090 / B, LAKE SHORE TOWERS RAJBHAVAN ROAD, SOMAJIGUDA, HYDERABAD- 500082 Telangana, INDI
Reliance Retail Limited 6- 3- 1090 / B, LAKE SHORE TOWERS RAJBHAVAN ROAD, SOMAJIGUDA, HYDERABAD- 500082 Telangana, INDI
Reliance Retail Limited FF101,Saffron Panchwati 5 Rasta,, Near Bank of Baroda Ambawadi Ahmedabad- 380006 Gujarat, INDIA
Reliance Retail Limited FF101,Saffron Panchwati 5 Rasta,, Near Bank of Baroda Ambawadi Ahmedabad- 380006 Gujarat, INDIA
Reliance Retail Limited NO 62/2,RIL BUILIDING RICHMOND ROAD, BANGALORE- 560025 Karnataka, INDIA SELLER PURCHASE ORDER V
Reliance Retail Limited NO 62/2,RIL BUILIDING RICHMOND ROAD, BANGALORE- 560025 Karnataka, INDIA SELLER PURCHASE ORDER V
Reliance Retail Limited NO 62/2,RIL BUILIDING RICHMOND ROAD, BANGALORE- 560025 Karnataka, INDIA SELLER PURCHASE ORDER V
Reliance Retail Limited NO 62/2,RIL BUILIDING RICHMOND ROAD, BANGALORE- 560025 Karnataka, INDIA SELLER PURCHASE ORDER V
Reliance Retail Limited NO 62/2,RIL BUIL

Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY 95, Anand Industrial Estate , Branch : 10 , Anand Industrial Estat
Page 1 of 3 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY 95, Anand Industrial Estate , Branch : 10 , Anand Industrial Estat
Page 1 of 3 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY 95, Anand Industrial Estate , Branch : 10 , Anand Industrial Estat
Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY 95, Anand Industrial Estate , Branch : 10 , Anand Industrial Estat
Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY 95, Anand Industrial Estate , Branch : 10 , Anand Industrial Estat
Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY 95, Anand Industrial Estate , Branch : 10 , Anand Industrial Estat
Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY 95, Anand Industrial Estate , Branch : 10 , Anand Industrial Estat
Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY 95, Anand Industrial Estate , Branch : 10 , Anand Industrial Estat
Page 1 of 2 TO / SHIP FROM: THE 

Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY GROUND FLOOR, Sy No689, opp Sai Geeta Ashramam, MEDCHAL ROAD OPP H
Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY GROUND FLOOR, Sy No689, opp Sai Geeta Ashramam, MEDCHAL ROAD OPP H
Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY GROUND FLOOR, Sy No689, opp Sai Geeta Ashramam, MEDCHAL ROAD OPP H
Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY GROUND FLOOR, Sy No689, opp Sai Geeta Ashramam, MEDCHAL ROAD OPP H
Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY GROUND FLOOR, Sy No689, opp Sai Geeta Ashramam, MEDCHAL ROAD OPP H
Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY GROUND FLOOR, Sy No689, opp Sai Geeta Ashramam, MEDCHAL ROAD OPP H
Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY Mehadia&Sons ,C&F Division,Gat No 1232 ,Opp.Fleet Guard Filters,Pu
Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY Mehadia&Sons ,C&F Division,Gat No 1232 ,Opp.Fleet Guard Filters,Pu
Page 1 of 2 TO / SHIP FROM: THE 

Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY Khata no 130/153, Khasra no 55/ 22/2(5-4), Sadhu Complex, Near Zir
Page 1 of 3 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY Khata no 130/153, Khasra no 55/ 22/2(5-4), Sadhu Complex, Near Zir
Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY Khata no 130/153, Khasra no 55/ 22/2(5-4), Sadhu Complex, Near Zir
Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY Khata no 130/153, Khasra no 55/ 22/2(5-4), Sadhu Complex, Near Zir
Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY Khata no 130/153, Khasra no 55/ 22/2(5-4), Sadhu Complex, Near Zir
Page 1 of 3 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY Khata no 130/153, Khasra no 55/ 22/2(5-4), Sadhu Complex, Near Zir
Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY Khata no 130/153, Khasra no 55/ 22/2(5-4), Sadhu Complex, Near Zir
Page 1 of 2 TO / SHIP FROM: THE HIMALAYA DRUG COMPANY Gondkhairy Village C/O,Mehadia & Sons, 18 Th Km Amaravarti Road Na


In [ ]:
import json

with open(file_name, "r", encoding='utf-8') as input_file:
    data = json.load(input_file)
    text = data[0]
    entities = data[0]['entities']
    
    classes = ['ORG', 'ADDRESS', 'GSTIN', 'PO NUMBER', 'PO DATE', "DELIVERY DATE", 'PO EXPIRY DATE', 'SUPPLIER NUMBER', 'PRODUCT DESCRIPTION', "PAN", "CIN", "QTY", "MRP"]
    annotation = {"classes":[cls for cls in classes],"annotations":[]}
    annotation['annotations'].append([text,{"entities": entities}])
    with open(f"PDF_annotated.json", "w", encoding='utf-8') as annot_file:
        json.dump(annotation, annot_file)